# 01 — Verificación real en la A100

Se ejecuta **después** del notebook 00 y **antes** de cualquier lote.

Genera 2 imágenes SDXL completas, una locución y un vídeo de prueba de 2 s: lo suficiente para saber que todo funciona, y barato en unidades.

| gate | qué comprueba |
|---|---|
| G0 | GPU, VRAM, torch, diffusers |
| G1 | perfil efectivo (SDXL full, 30 pasos, hires, threads) |
| G2 | imagen final == canvas 1080x1920, y el coste real por imagen |
| G3 | voz `es-MX-JorgeNeural` masterizada a -14 LUFS |
| G4 | MP4 1080x1920 con audio y subtítulos |

Si algo falla, la celda se pone en rojo y **no hay que lanzar el lote**.


In [ ]:
#@title 0. Preflight de la sesión { display-mode: "form" }
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/canal-estoicismo")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import scripts.colab_env as env

if not (env.PROJECT_DIR / ".git").exists() or not (env.PROJECT_DIR / "output").exists():
    raise SystemExit("Sesión nueva o incompleta: ejecuta primero 00_entorno_A100.ipynb.")

commit = subprocess.run(
    ["git", "-C", str(env.PROJECT_DIR), "log", "-1", "--oneline"],
    capture_output=True, text=True,
).stdout.strip()
print("proyecto:", env.PROJECT_DIR)
print("commit  :", commit)


In [ ]:
#@title 1. GATE — aceptación end-to-end { display-mode: "form" }
code = env.run(
    [sys.executable, "scripts/colab_check.py", "--format", "reel"],
    cwd=env.PROJECT_DIR,
)
env.gate(
    "aceptación A100",
    code == 0,
    "corrige los [FAIL] de arriba antes de gastar unidades en un lote",
)


In [ ]:
#@title 2. Coste medido y proyección del lote { display-mode: "form" }
import json

report_path = env.PROJECT_DIR / "output" / "_checks" / "acceptance.json"
if not report_path.exists():
    raise SystemExit("No hay informe de aceptación: la celda 1 no llegó a terminar bien.")

report = json.loads(report_path.read_text(encoding="utf-8"))
gates = {g["name"]: g["seconds"] for g in report.get("gates", [])}

warm = report.get("warm_image_seconds") or 0.0
cold = report.get("cold_image_seconds") or 0.0
tts_seconds = gates.get("Voice") or 0.0

canvas = report["canvas"]
print(f"GPU                : {report['device']} ({report['vram_gb']} GB)")
print(f"canvas             : {canvas[0]}x{canvas[1]}")
print(f"imagen en frío     : {cold:.1f} s  (incluye cargar los modelos: se paga una vez)")
print(f"imagen en caliente : {warm:.1f} s  <- el coste real por escena")
print(f"voz (una frase)    : {tts_seconds:.1f} s")

SCENES = 10
print(f"\nProyección para un reel de {SCENES} escenas:")
print(f"  imágenes    : ~{warm * SCENES / 60:.1f} min")
print(f"  voz         : ~{tts_seconds * SCENES / 60:.1f} min")
print("  composición : ~6-12 min (libx264 en 12 vCPU, aproximadamente 1x-2x tiempo real)")
print(f"  TOTAL       : ~{(warm * SCENES + 60 * 9) / 60:.0f} min por reel")

print("\nLas imágenes se cachean en Drive: repetir un reel no las vuelve a pagar.")


## Decisión

- **Todo en `[PASS]`** → adelante con `02_lote_reels.ipynb`.
- **Algún `[FAIL]`** → no lances el lote. Corrige eso primero.

### Qué significa cada fallo típico

| fallo | causa más probable | arreglo |
|---|---|---|
| G0 sin CUDA | runtime sin GPU o reinicio pendiente | Entorno de ejecución > A100 GPU, y reejecutar 00 |
| G2 tamaño ≠ canvas | perfil no aplicado | revisa que el doctor diga `--profile a100` |
| G2 `hires=NO` | falló la carga del pipeline img2img | vuelve a ejecutar: suele ser un fallo de red al descargar |
| G3 LUFS fuera de rango | cadena de audio alterada | revisa `src/audio/tts_engine.py` |
| G4 sin audio | ffmpeg sin códec aac | reejecuta la instalación de la celda 6 del notebook 00 |
